# Split and move the DPR processing flow

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-672

In [ ]:
# Deploy flow from the rs-client-libraries git repo or using the bucket ?
# The goal is to use the git repo and "develop" branch but it causes errors
# when we make changes in a new branch.
deploy_with_bucket = True

In [ ]:
debug_flow = False # For testing only. Should be False in git.
if debug_flow:
    deploy_with_bucket = True

## 1. Initialisation

In [ ]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
USE_DPR_MOCKUP = True

# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_mockup(scale=2)
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)
display(dask_cluster_staging)

In [ ]:
# Create a test collection
CATALOG_COLLECTION_ID = "SPRINT24_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)

# Check that it is empty
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
assert not list(items)

# Other test values
SESSION_ID = "S1A_20200105072204051312"
CADIP_COLLECTION_ID = "sgs_sentinel1"

In [ ]:
# Other imports
from importlib import reload
import os
import os.path as osp
import sys
import prefect
from pystac import ItemCollection
from rs_common import prefect_utils
from rs_common.prefect_utils import *
import rs_workflows
from rs_workflows.flow_utils import ProcessorEnum
import resources

# Local paths
rs_workflows_parent = Path(rs_workflows.__path__[0]).parent

# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

In [ ]:
main_flow_parameters = {
  "env": {
    "owner_id": OWNER_ID,
  },
  "processor": ProcessorEnum.S3L0,
  "cadip_collection_identifier": CADIP_COLLECTION_ID,
  "session_identifier": SESSION_ID,
  "catalog_collection_identifier": CATALOG_COLLECTION_ID,
  "s3_payload_template": osp.join(
      s3_config, 
      "s3/s3_l0_demo_payload_dpr_mockup_template.yaml"
  ),
  "s3_output_data": f"{s3_output}/s3",
  "use_dpr_mockup": True,
}

cadip_search_parameters = {
  "env": {
    "owner_id": OWNER_ID,
  },
  "cadip_collection_identifier": CADIP_COLLECTION_ID,
  "session_identifier": SESSION_ID
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

## 2. Deploy Prefect flow

We deploy the Prefect workflows that are implemented in the `rs-client-libraries` git repository.

WARNING: the `rs-client-libraries` source code must be identical in these 3 environments:

  * https://github.com/RS-PYTHON/rs-demo.git
  * This Jupyter environment
  * The Prefect Docker images

In [ ]:
%%bash -s "$rs_workflows_parent"
# Deploy the flows
deploy_file=$(realpath "./split_processor_flow.yaml")
echo "Deploying '$deploy_file'..."
(cd $1; prefect --no-prompt deploy --prefect-file "$deploy_file" --all)

In [ ]:
# Flow deployment names
main_deploy = "On-demand processing/On-demand processing"
cadip_deploy = "Cadip search/Cadip search"
auxip_deploy = "Auxip search/Auxip search"
staging_deploy = "Staging/Staging"

# Wait for deployments
for deploy_name in [main_deploy, cadip_deploy, auxip_deploy, staging_deploy]:
    await prefect_utils.wait_for_deployment(deploy_name)

In [ ]:
# For testing only: serve from a s3 bucket to test changes more easily
if deploy_with_bucket:

    # Use a subfolder named after the current user
    s3_code_folder = f"users/{OWNER_ID}/code" 
    workflows_folder = f"{s3_code_folder}/rs_workflows"
    
    # Upload workflows package and resources contents
    await share_bucket.put_directory(local_path = rs_workflows.__path__[0], to_path = workflows_folder)
    await share_bucket.put_directory(local_path = resources.__path__[0], to_path = f"{s3_code_folder}/resources")

    # Reload all rs-client-libraries modules
    for module in list(sys.modules.values()):
        if any(module.__name__.startswith(prefix) for prefix in ["rs_client.", "rs_common.", "rs_workflows."]):
            reload(module)

    # Deploy the flows
    for entrypoint, name, deploy_name in [
        ["on_demand_processing.py:on_demand_processing", "On-demand processing",main_deploy],
        ["auxip_flow.py:search", "Auxip search", auxip_deploy],
        ["cadip_flow.py:search", "Cadip search", cadip_deploy],
        ["staging_flow.py:staging", "Staging", staging_deploy],
    ]:
        flow = await prefect.flow.from_source(
            source=share_bucket,
            entrypoint=f"{workflows_folder}/{entrypoint}",
        )
        await flow.deploy(
            name=name,
            work_pool_name=os.environ["PREFECT_WORK_POOL_EOPF"], 
            tags=["debug only"],
            ignore_warnings=True,
        )
        await prefect_utils.wait_for_deployment(deploy_name)

## 3. Run the main Prefect flow

In [ ]:
# Convert to json to trigger prefect flow
params_str = to_json(main_flow_parameters)

<div class="alert alert-block alert-danger">
WARNING: if we deploy using the git repo, by default we use the 'develop' branch from rs-client-libraries, see split_processor_flow.yaml
</div>

In [ ]:
%%bash -s "$main_deploy" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

In [ ]:
# Processed items published to the catalog
ItemCollection(list(catalog_client.get_items(CATALOG_COLLECTION_ID)))

## 4. We can also run only a subflow

In [ ]:
# Convert to json to trigger prefect flow
cadip_params_str = to_json(cadip_search_parameters)

In [ ]:
%%bash -s "$cadip_deploy" "$cadip_params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

<div class="alert alert-block alert-warning">

Note: how to get the job results ?
</div>

## Shutdown the dask clusters

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)
    dask_gateway_staging.scale_cluster(dask_cluster_staging.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)
    shutdown_dask_clusters(dask_gateway_staging, dask_cluster_staging.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [ ]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway_staging, None)
    shutdown_dask_clusters(dask_gateway_eopf, None)
    init_dask_cluster_mockup(scale=2)
    init_dask_cluster_staging(scale=2)

    from resources.dask_utils import *

In [ ]:
if debug_flow:

    # Reload all rs-client-libraries modules
    for module in list(sys.modules.values()):
        if any(module.__name__.startswith(prefix) for prefix in ["rs_client.", "rs_common.", "rs_workflows."]):
            reload(module)

    from rs_workflows import on_demand_processing
    results = await on_demand_processing.on_demand_processing(**main_flow_parameters)
    display(results)